In [11]:
import os 

In [12]:
%pwd

'e:\\Text-Summarizer-Project'

In [13]:
os.chdir('e:\\Text-Summarizer-Project')

In [14]:
%pwd

'e:\\Text-Summarizer-Project'

In [15]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)

class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [16]:
import sys
# This tells Python to look for modules inside the 'src' folder
sys.path.append("src")
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories



In [17]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, params_file_path: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(

            root_dir = config.root_dir,
            data_path = config.data_path,
            model_model_path = config.model_model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
        )
    
        
        return model_evaluation_config

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset, load_from_disk
import evaluate
import torch
import pandas as pd
from tqdm import tqdm

e:\anaconda\anaconda_nav\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split the dataset into smaller batches that we can process simultaneously.
        Yields successive batch-sized chunks from list_of_elements.
        """
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]


    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="article",
        column_summary="highlights"
    ):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):
            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )

            # Decode generated summaries
            decoded_summaries = [
                tokenizer.decode(
                    summary,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for summary in summaries
            ]

            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]


            # Add predictions and references to the metric
            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )

        # Compute final score
        score = metric.compute()
        return score
        



        
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(self.config.tokenizer_path)
        print(type(self.config.tokenizer_path))
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
    
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_model_path).to(device)
       
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)


        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
  
        rouge_metric = evaluate.load('rouge')

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt['test'][0:100], rouge_metric, model_pegasus, tokenizer, batch_size=2, column_text='dialogue', column_summary='summary'
        )

        rouge_dict = dict((rn, score[rn]) for rn in rouge_names)

        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file_name, index=False)



    def summarize_custom_text(self, text: str):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
    
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_model_path).to(device)

        inputs = tokenizer(
            text,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        summaries = model_pegasus.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        # Decode generated summaries
        decoded_summaries = [
            tokenizer.decode(
                summary,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )
            for summary in summaries
        ]

    

        return decoded_summaries[0]

        

        





        

        

In [26]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
NVIDIA GeForce RTX 4060 Laptop GPU


In [30]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e:
    raise e

[2026-05-24 00:04:33,630: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-05-24 00:04:33,632: INFO: common]: yaml file: params.yaml loaded successfully
[2026-05-24 00:04:33,633: INFO: common]: created directory at: artifacts
[2026-05-24 00:04:33,634: INFO: common]: created directory at: artifacts/model_evaluation
artifacts/model_trainer/tokenizer
<class 'str'>


100%|██████████| 50/50 [02:54<00:00,  3.50s/it]

[2026-05-24 00:07:35,860: INFO: rouge_scorer]: Using default tokenizer.


In [ ]:
import torch

# Check if CUDA is available
print(f"Is CUDA available: {torch.cuda.is_available()}")

# Get the name of the GPU
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Current device: {torch.cuda.current_device()}")
else:
    print("CUDA is not available. Training will run on CPU.")

In [31]:
model_evaluation.summarize_custom_text("Hello, how are you doing today? I hope everything is going well. I wanted to check in and see if you have any plans for the weekend. Let me know if you'd like to catch up sometime soon!")

" I   w a n t e d   t o   c h e c k   i n   a n d   s e e   i f   y o u   h a v e   a n y   p l a n s   f o r   t h e   w e e k e n d .   L e t   m e   k n o w   i f   y o u ' d   l i k e   t o   c a t c h   u p   s o m e t i m e   s o o n ,   t o o . "